# Lorenz sampling training with multiple random seeds

This notebook mirrors `exmaples/lorenz/train_lorenz_sampling.py` but runs multiple training
experiments using different random seeds.

In [1]:
import os
import sys
import datetime
import random
import numpy as np
import pandas as pd
import tensorflow as tf

sys.path.append("../../src")

from example_lorenz import get_lorenz_data
from sindy_utils import library_size
from training_lorenz import train_network

from tensorflow.python.client import device_lib

device_lib.list_local_devices()
print(device_lib.list_local_devices())


/home/szupernikusz/miniconda3/envs/mars/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/szupernikusz/miniconda3/envs/mars/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/szupernikusz/miniconda3/envs/mars/lib/python3.7/site-packages/tensorflow/python/framework/dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/szupernikusz/miniconda3/e

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 9599564809466585278
, name: "/device:XLA_CPU:0"
device_type: "XLA_CPU"
memory_limit: 17179869184
locality {
}
incarnation: 2140526847350604275
physical_device_desc: "device: XLA_CPU device"
, name: "/device:XLA_GPU:0"
device_type: "XLA_GPU"
memory_limit: 17179869184
locality {
}
incarnation: 15211550292443080985
physical_device_desc: "device: XLA_GPU device"
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 6917246157
locality {
  bus_id: 1
  links {
  }
}
incarnation: 11329736990645956731
physical_device_desc: "device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 12.0"
]


2026-01-29 01:15:11.895357: E tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:991] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 01:15:11.896404: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x5dd63b2021f0 executing computations on platform CUDA. Devices:
2026-01-29 01:15:11.896418: I tensorflow/compiler/xla/service/service.cc:175]   StreamExecutor device (0): NVIDIA GeForce RTX 5060 Laptop GPU, Compute Capability 12.0
2026-01-29 01:15:11.898912: E tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:991] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-01-29 01:15:11.898941: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1640] Found device 0 with properties: 
name: NVIDIA GeForce RTX 5060 Laptop GPU major: 12 minor: 0 memoryClockRate(GHz): 1.56
pciBusID: 0000:02:00.0


## Generate training data
This uses the same data generation logic as the training script.

In [ ]:
noise_strength = 1e-6

print("Start of data generation")
training_data = get_lorenz_data(1024, noise_strength=noise_strength)
validation_data = get_lorenz_data(20, noise_strength=noise_strength)
print("End of data generation")

print("Sindy Coefficient generated")
print(training_data['sindy_coefficients'])


## Configure training parameters
Parameters match the script defaults.

In [ ]:
params = {}

params['input_dim'] = 128
params['latent_dim'] = 3
params['model_order'] = 1
params['poly_order'] = 3
params['include_sine'] = False
params['library_dim'] = library_size(
    params['latent_dim'],
    params['poly_order'],
    params['include_sine'],
    True,
)

# sequential thresholding parameters
params['sequential_thresholding'] = True
params['coefficient_threshold'] = 0.1
params['threshold_frequency'] = 500
params['threshold_start'] = 0
params['coefficient_mask'] = np.ones((params['library_dim'], params['latent_dim']))
params['nonactive_counter'] = np.zeros((params['library_dim'], params['latent_dim']))
params['coefficient_initialization'] = 'constant'

# loss function weighting
params['loss_weight_decoder'] = 1.0
params['loss_weight_sindy_z'] = 0.0
params['loss_weight_sindy_x'] = 1e-4

params['activation'] = 'sigmoid'
params['widths'] = [64, 32]

# training parameters
params['epoch_size'] = training_data['x'].shape[0]
params['batch_size'] = 1024

params['data_path'] = os.getcwd() + '/'
params['print_progress'] = True
params['print_frequency'] = 50

# Bayesian parameters
params['learning_rate'] = 1e-3
params['prior'] = 'laplace'
params['loss_weight_sindy_regularization'] = 1e-5
params['pi'] = 0.116
params['c_std'] = 20000000.0
params['epsilon'] = 0.2
params['decay'] = 0.02
params['sigma'] = 1.0

# training time cutoffs
params['max_epochs'] = 5001
params['refinement_epochs'] = 1001

print(tf.__version__)


## Train multiple models with different seeds
Update the `seeds` list to run more or fewer experiments.

In [ ]:
seeds = [0, 1, 2]
results = []

for seed in seeds:
    print(f'EXPERIMENT seed={seed}')

    random.seed(seed)
    np.random.seed(seed)
    tf.set_random_seed(seed)

    params['coefficient_mask'] = np.ones((params['library_dim'], params['latent_dim']))
    params['save_name'] = (
        'lorenz_seed_'
        + str(seed)
        + '_'
        + datetime.datetime.now().strftime('%Y_%m_%d_%H_%M_%S_%f')
    )

    tf.reset_default_graph()

    results_dict = train_network(training_data, validation_data, params)

    print('training finished')

    results.append({**results_dict, **params, 'seed': seed})

df = pd.DataFrame(results)
df.to_pickle(
    'experiment_results_'
    + datetime.datetime.now().strftime('%Y%m%d%H%M')
    + '.pkl'
)
df
